In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Project 1: Explainable AI in a Radiology Scenario

In [ ]:
%pip install opencv-python

In [39]:
import zipfile
import os
import cv2
import pandas as pd
import torch
import kagglehub
from sklearn.model_selection import train_test_split

In [27]:
DOWNLOAD_DATA = False
OUTPUT_DIR = "/content/drive/MyDrive/XAI_Group4/frontal_resized/"

## Part 1: Data Acquisition and Preprocessing

In [22]:
# Function to resize images using OpenCV
def preprocess_image(input_path, output_path, size=(224, 224)):
    try:
        # Read the image using OpenCV
        image = cv2.imread(input_path)
        if image is None:
            print(f"Failed to read image: {input_path}")
            return

        # Resize the image
        resized_image = cv2.resize(image, size)
        normalized_image = cv2.normalize(resized_image, None, alpha=0)

        # Save the resized image
        cv2.imwrite(output_path, resized_image)
    except Exception as e:
        print(f"Failed to resize image {input_path}: {e}")

def get_data_from_kaggle(output_path):
  # Directory where the dataset is located
  input_dir = kagglehub.dataset_download("ashery/chexpert")  # Path returned from kagglehub.dataset_download

  # Create the output directory if it doesn't exist
  os.makedirs(output_path, exist_ok=True)

  # Traverse through the dataset and resize all images
  for dir, _, files in os.walk(input_dir):
      for file in files:
          if file.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
              input_image_path = os.path.join(dir, file)

              # Create the same subdirectory structure in the output folder
              relative_path = os.path.relpath(input_image_path, input_dir)
              output_image_path = os.path.join(output_path, relative_path)
              os.makedirs(os.path.dirname(output_image_path), exist_ok=True)

              # Resize and save the image
              preprocess_image(input_image_path, output_image_path)
          # store the csv files
          if file.endswith('.csv'):
            csv = pd.read_csv(os.path.join(dir, file))
            csv.to_csv(os.path.join(output_path, file))

  print(f"All images resized and saved in: {output_path}")


In [25]:
if DOWNLOAD_DATA:
  get_data_from_kaggle(OUTPUT_DIR)

Resuming download from 452984832 bytes (11043145677 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/ashery/chexpert?dataset_version_number=1 (452984832/11496130509) bytes left.


100%|██████████| 10.7G/10.7G [02:49<00:00, 65.3MB/s]

Extracting files...


All images resized and saved in: /content/drive/MyDrive/XAI_Group4/frontal_resized/


In [43]:
def load_frontal_from_csv(path):
  df = pd.read_csv(path)
  df = df.loc[df['Frontal/Lateral'] == "Frontal"] # only include frontal data
  return df

In [44]:
# load data into dataframes
df_all_train = load_frontal_from_csv(os.path.join(OUTPUT_DIR,"train.csv"))
df_valid = load_frontal_from_csv(os.path.join(OUTPUT_DIR,"valid.csv"))

# split the train data
X = df_all_train.drop('Pleural Effusion', axis=1)
y = df_all_train['Pleural Effusion']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Part 2: Model creation, training and testing
DenseNet model


## Part 3: Explaining the model